In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import umap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [20]:
X_train_true = pd.read_csv("X_train_true.csv", header = None).to_numpy()
missing_mask_train = pd.read_csv("missing_mask_train.csv", header = None).to_numpy()
X_test_true = pd.read_csv("X_test_true.csv", header = None).to_numpy()
missing_mask_test = pd.read_csv("missing_mask_test.csv", header = None).to_numpy()

sensitive_train = pd.read_csv("sensitive_train.csv", header = None).to_numpy()
sensitive_test = pd.read_csv("sensitive_test.csv", header = None).to_numpy()

In [21]:
X_train_miss = X_train_true
X_train_miss[missing_mask_train] = np.nan

X_test_miss = X_test_true
X_test_miss[missing_mask_test] = np.nan

In [22]:
class TabularDataset(Dataset):
    def __init__(self, X, s, nan_value=np.nan):
        """
        X: numpy array or tensor of shape (N, D)
        s: numpy array or tensor of shape (N,) with 0/1 sensitive attribute
        nan_value: how missing values are encoded, e.g. np.nan
        """
        X = np.asarray(X, dtype=np.float32)
        s = np.asarray(s, dtype=np.int64)

        # mask: 1 for observed, 0 for missing
        if np.isnan(nan_value):
            mask = (~np.isnan(X)).astype(np.float32)
            X = np.nan_to_num(X, nan=0.0)
        else:
            mask = (X != nan_value).astype(np.float32)
            X = np.where(mask == 1, X, 0.0).astype(np.float32)

        self.X = torch.from_numpy(X)
        self.mask = torch.from_numpy(mask)
        self.s = torch.from_numpy(s)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return {
            "x": self.X[idx],       # (D,)
            "mask": self.mask[idx], # (D,)
            "s": self.s[idx],       # scalar 0/1
        }

In [34]:
class ConditionalVAE(nn.Module):
    def __init__(
        self,
        input_dim=100,
        latent_dim=16,
        hidden_dim=64,
        cond_dim=2,        # one-hot for binary sensitive attribute
        beta_kl=1.0,
        lambda_fair=1.0,
        mmd_kernel_bandwidth=1.0,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim
        self.beta_kl = beta_kl
        self.lambda_fair = lambda_fair
        self.mmd_kernel_bandwidth = mmd_kernel_bandwidth

        # Encoder: [x * mask, mask, s_onehot]
        enc_input_dim = input_dim * 2 + cond_dim     # x*mask, mask, s_onehot
        self.encoder_layer = nn.Linear(enc_input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder: [z, s_onehot] -> x_hat
        dec_input_dim = latent_dim + cond_dim
        self.decoder_layer = nn.Linear(dec_input_dim, hidden_dim)
        self.decoder_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x, mask, s):
        """
        x: (B, D)
        mask: (B, D)
        s: (B,) binary
        """
        s = s.view(-1).long()
        s_onehot = F.one_hot(s, num_classes=self.cond_dim).float()
        enc_input = torch.cat([x * mask, mask, s_onehot], dim=-1)
        h = F.relu(self.encoder_layer(enc_input))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, s):
        s = s.view(-1).long()
        s_onehot = F.one_hot(s, num_classes=self.cond_dim).float()
        dec_input = torch.cat([z, s_onehot], dim=-1)
        h = F.relu(self.decoder_layer(dec_input))
        x_hat = self.decoder_out(h)
        return x_hat

    def forward(self, x, mask, s):
        mu, logvar = self.encode(x, mask, s)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, s)
        return x_hat, mu, logvar, z

    # ---------- Loss components ---------- #

    def reconstruction_loss(self, x, x_hat, mask):
        """
        MSE over observed entries only.
        """
        diff2 = (x_hat - x) ** 2
        diff2 = diff2 * mask
        # avoid division by zero
        denom = mask.sum().clamp(min=1.0)
        return diff2.sum() / denom

    def kl_loss(self, mu, logvar):
        # standard KL between q(z|x) and N(0, I)
        kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
        return kl.mean()

    def _rbf_kernel(self, x, y):
        """
        x: (Nx, D), y: (Ny, D)
        RBF kernel with fixed bandwidth.
        """
        # ||x - y||^2
        diff = x.unsqueeze(1) - y.unsqueeze(0)   # (Nx, Ny, D)
        dist_sq = (diff ** 2).sum(-1)           # (Nx, Ny)
        k = torch.exp(-dist_sq / (2 * self.mmd_kernel_bandwidth ** 2))
        return k

    def mmd_fairness_loss(self, z, s):
        """
        Encourage z to be similar across s=0 and s=1 (independence).
        MMD^2 between latent distributions of two groups.
        """
        s = s.view(-1)
        z0 = z[s == 0]
        z1 = z[s == 1]

        # If batch has only one group, skip fairness penalty
        if z0.shape[0] < 2 or z1.shape[0] < 2:
            return z.new_tensor(0.0)

        K00 = self._rbf_kernel(z0, z0)
        K11 = self._rbf_kernel(z1, z1)
        K01 = self._rbf_kernel(z0, z1)

        mmd = K00.mean() + K11.mean() - 2 * K01.mean()
        return mmd

    def loss(self, x, mask, s, x_hat, mu, logvar, z):
        recon = self.reconstruction_loss(x, x_hat, mask)
        kl = self.kl_loss(mu, logvar)
        mmd = self.mmd_fairness_loss(z, s)

        total = recon + self.beta_kl * kl + self.lambda_fair * mmd

        loss_dict = {
            "loss": total,
            "recon": recon.detach(),
            "kl": kl.detach(),
            "mmd": mmd.detach(),
        }
        return total, loss_dict

    # ---------- Imputation ---------- #

    @torch.no_grad()
    def impute(self, x, mask, s, n_samples=10):
        """
        Impute missing entries of x, conditioned on s.
        x, mask, s: tensors on same device.
        x: (B, D), mask: (B, D), s: (B,)
        """
        self.eval()
        B, D = x.shape
        x = x.clone()
        mask = mask.clone()
        s = s.clone()

        # multiple samples for smoother imputation
        recon_accum = torch.zeros_like(x)
        for _ in range(n_samples):
            x_hat, mu, logvar, z = self.forward(x, mask, s)
            recon_accum += x_hat

        x_hat_mean = recon_accum / n_samples

        # Fill only missing entries: where mask == 0
        imputed = x * mask + x_hat_mean * (1 - mask)
        return imputed


In [35]:
def train_cvae(
    X,
    s,
    batch_size=128,
    lr=1e-3,
    num_epochs=50,
    latent_dim=16,
    hidden_dim=64,
    beta_kl=1.0,
    lambda_fair=1.0,
    device="cuda" if torch.cuda.is_available() else "cpu",
):
    dataset = TabularDataset(X, s)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    input_dim = dataset.X.shape[1]
    model = ConditionalVAE(
        input_dim=input_dim,
        latent_dim=latent_dim,
        hidden_dim=hidden_dim,
        beta_kl=beta_kl,
        lambda_fair=lambda_fair,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in loader:
            x = batch["x"].to(device)
            mask = batch["mask"].to(device)
            s_batch = batch["s"].to(device)

            optimizer.zero_grad()
            x_hat, mu, logvar, z = model(x, mask, s_batch)
            loss, loss_dict = model.loss(x, mask, s_batch, x_hat, mu, logvar, z)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)

        epoch_loss /= len(dataset)
        print(
            f"Epoch {epoch:03d} | loss={epoch_loss:.4f} "
            f"(recon={loss_dict['recon']:.4f}, kl={loss_dict['kl']:.4f}, mmd={loss_dict['mmd']:.4f})"
        )

    return model

In [43]:
# Suppose you already have X, s and train the model:
model = train_cvae(X_train_miss, sensitive_train, num_epochs=300, lambda_fair=10.0)

# Build dataset again to get X, mask tensors
dataset_train = TabularDataset(X_train_miss, sensitive_train)
X_tensor_train = dataset_train.X
mask_tensor_train = dataset_train.mask
s_tensor_train = dataset_train.s

device = next(model.parameters()).device
X_tensor_train = X_tensor_train.to(device)
mask_tensor_train = mask_tensor_train.to(device)
s_tensor_train = s_tensor_train.to(device)

with torch.no_grad():
    X_imputed_train = model.impute(X_tensor_train, mask_tensor_train, s_tensor_train, n_samples=20)
    
dataset_test = TabularDataset(X_test_miss, sensitive_test)
X_tensor_test = dataset_test.X
mask_tensor_test = dataset_test.mask
s_tensor_test = dataset_test.s

device = next(model.parameters()).device
X_tensor_test = X_tensor_test.to(device)
mask_tensor_test = mask_tensor_test.to(device)
s_tensor_test = s_tensor_test.to(device)

with torch.no_grad():
    X_imputed_test = model.impute(X_tensor_test, mask_tensor_test, s_tensor_test, n_samples=20)

X_imputed_train = X_imputed_train.cpu().numpy()
X_imputed_test = X_imputed_test.cpu().numpy()

Epoch 001 | loss=1.7908 (recon=1.2171, kl=0.0169, mmd=0.0822)
Epoch 002 | loss=1.7303 (recon=1.2905, kl=0.0084, mmd=0.0855)
Epoch 003 | loss=1.7084 (recon=1.2472, kl=0.0055, mmd=0.0852)
Epoch 004 | loss=1.6875 (recon=1.1310, kl=0.0038, mmd=0.0980)
Epoch 005 | loss=1.6734 (recon=1.1626, kl=0.0036, mmd=0.0798)
Epoch 006 | loss=1.6684 (recon=1.1569, kl=0.0041, mmd=0.1116)
Epoch 007 | loss=1.6441 (recon=1.1729, kl=0.0051, mmd=0.0887)
Epoch 008 | loss=1.6405 (recon=1.1028, kl=0.0044, mmd=0.0983)
Epoch 009 | loss=1.6200 (recon=1.1561, kl=0.0073, mmd=0.0819)
Epoch 010 | loss=1.6178 (recon=1.0807, kl=0.0094, mmd=0.0770)
Epoch 011 | loss=1.6169 (recon=1.1041, kl=0.0108, mmd=0.1114)
Epoch 012 | loss=1.6174 (recon=1.1127, kl=0.0140, mmd=0.1043)
Epoch 013 | loss=1.6073 (recon=1.1118, kl=0.0154, mmd=0.0771)
Epoch 014 | loss=1.5972 (recon=1.1042, kl=0.0186, mmd=0.0981)
Epoch 015 | loss=1.5872 (recon=1.0596, kl=0.0186, mmd=0.0793)
Epoch 016 | loss=1.6057 (recon=1.0611, kl=0.0237, mmd=0.1200)
Epoch 01

Epoch 144 | loss=1.5533 (recon=0.9993, kl=0.0440, mmd=0.1115)
Epoch 145 | loss=1.5694 (recon=1.0103, kl=0.0355, mmd=0.1307)
Epoch 146 | loss=1.5570 (recon=1.0067, kl=0.0395, mmd=0.1039)
Epoch 147 | loss=1.5474 (recon=1.0165, kl=0.0394, mmd=0.0984)
Epoch 148 | loss=1.5452 (recon=1.0387, kl=0.0399, mmd=0.0931)
Epoch 149 | loss=1.5433 (recon=1.0142, kl=0.0372, mmd=0.0936)
Epoch 150 | loss=1.5363 (recon=1.0027, kl=0.0304, mmd=0.0767)
Epoch 151 | loss=1.5494 (recon=1.0003, kl=0.0404, mmd=0.0924)
Epoch 152 | loss=1.5644 (recon=1.0146, kl=0.0373, mmd=0.1205)
Epoch 153 | loss=1.5523 (recon=0.9539, kl=0.0401, mmd=0.1039)
Epoch 154 | loss=1.5586 (recon=0.9994, kl=0.0416, mmd=0.0982)
Epoch 155 | loss=1.5512 (recon=0.9572, kl=0.0396, mmd=0.0982)
Epoch 156 | loss=1.5569 (recon=1.0285, kl=0.0464, mmd=0.1209)
Epoch 157 | loss=1.5479 (recon=0.9915, kl=0.0432, mmd=0.1043)
Epoch 158 | loss=1.5539 (recon=1.0027, kl=0.0415, mmd=0.1036)
Epoch 159 | loss=1.5394 (recon=0.9890, kl=0.0399, mmd=0.0795)
Epoch 16

Epoch 287 | loss=1.5420 (recon=1.0257, kl=0.0403, mmd=0.1041)
Epoch 288 | loss=1.5408 (recon=0.9764, kl=0.0373, mmd=0.0981)
Epoch 289 | loss=1.5463 (recon=0.9914, kl=0.0403, mmd=0.1114)
Epoch 290 | loss=1.5504 (recon=1.0018, kl=0.0406, mmd=0.1042)
Epoch 291 | loss=1.5421 (recon=0.9847, kl=0.0386, mmd=0.0819)
Epoch 292 | loss=1.5424 (recon=0.9970, kl=0.0411, mmd=0.0790)
Epoch 293 | loss=1.5413 (recon=1.0147, kl=0.0375, mmd=0.0932)
Epoch 294 | loss=1.5396 (recon=0.9968, kl=0.0394, mmd=0.0821)
Epoch 295 | loss=1.5442 (recon=1.0037, kl=0.0381, mmd=0.1041)
Epoch 296 | loss=1.5586 (recon=1.0034, kl=0.0384, mmd=0.1041)
Epoch 297 | loss=1.5570 (recon=0.9914, kl=0.0386, mmd=0.0980)
Epoch 298 | loss=1.5437 (recon=1.0007, kl=0.0386, mmd=0.0794)
Epoch 299 | loss=1.5454 (recon=0.9995, kl=0.0397, mmd=0.0854)
Epoch 300 | loss=1.5526 (recon=1.0044, kl=0.0415, mmd=0.0889)


In [45]:
pd.DataFrame(X_imputed_train).to_csv("/uufs/chpc.utah.edu/common/home/u1417165/cVAE_imputed_train.csv",header=False, index=False)
pd.DataFrame(X_imputed_test).to_csv("/uufs/chpc.utah.edu/common/home/u1417165/cVAE_imputed_test.csv",header=False, index=False)